# merge-trades

The purpose of this example is to demonstrate the merge capability of the trade log.  You may want to merge your trades when you are examining the result of trading algorithms that scaling in and/or out of positions.  You can view a whole sequence of trades as one trade instead of the individual trades.  For example, you scale in 4 different buy operations, then sell all at once.  Using merge, those 4 trades will be grouped into 1 buy operation with the single sell operation.  Statistical Analysis in pinkfish will treat this as a single trade instead of 4 distinct trades.  This will be useful in evaluating the effectiveness of a scaling algorithm.

    Double 7's (Short Term Trading Strategies that Work)

    1. The SPY is above its 200-day moving average
    2. The SPY closes at a X-day low, buy some shares.
       If it falls further, buy some more, etc...
    3. If the SPY closes at a X-day high, sell your entire long position.
    
    (Scaling in; compare regular trade log vs merged trade log)

In [1]:
import datetime

import matplotlib.pyplot as plt
import pandas as pd

import pinkfish as pf
import strategy

# Format price data.
pd.options.display.float_format = '{:0.2f}'.format

%matplotlib inline

In [2]:
# Set size of inline plots.
'''note: rcParams can't be in same cell as import matplotlib
   or %matplotlib inline
   
   %matplotlib notebook: will lead to interactive plots embedded within
   the notebook, you can zoom and resize the figure
   
   %matplotlib inline: only draw static images in the notebook
'''
plt.rcParams["figure.figsize"] = (10, 7)

Some global data

In [3]:
capital = 10000
start = datetime.datetime(2020, 1, 1)
end = datetime.datetime.now()

Define symbols

In [4]:
symbols = ['SPY', 'SPY_merged']

Options

In [5]:
options = {
    'use_adj' : False,
    'use_cache' : True,
    'margin' : 1,
    'period' : 7,
    'max_open_trades' : 4,
    'enable_scale_in' : True,
    'enable_scale_out' : True,
    'merge_trades' : False
}

Run Strategy

In [6]:
options['merge_trades'] = False
strategies = pd.Series(dtype=object)
for symbol in symbols:
    print(symbol, end=" ")
    strategies[symbol] = strategy.Strategy(symbols[0], capital, start, end, options)
    strategies[symbol].run()
    options['merge_trades'] = True

SPY SPY_merged 

In [7]:
# View all columns and row in dataframe
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# View raw log
strategies[symbols[0]].rlog

,date,seq_num,price,shares,entry_exit,direction,symbol
0,2020-01-27,0,323.50,7,entry,LONG,SPY
1,2020-01-31,1,321.73,7,entry,LONG,SPY
2,2020-02-04,2,329.06,8,exit,LONG,SPY
3,2020-02-05,3,332.86,6,exit,LONG,SPY
4,2020-02-21,4,333.48,7,entry,LONG,SPY
5,2020-02-24,5,322.42,7,entry,LONG,SPY
6,2020-02-25,6,312.65,7,entry,LONG,SPY
7,2020-02-26,7,311.50,7,entry,LONG,SPY
8,2020-03-04,8,312.86,9,exit,LONG,SPY
9,2020-03-26,9,261.20,9,exit,LONG,SPY


In [8]:
# View unmerged trade log
strategies[symbols[0]].tlog

,entry_date,entry_price,exit_date,exit_price,pl_points,pl_cash,qty,cumul_total,direction,symbol
0,2020-01-27,323.50,2020-02-04,329.06,5.56,38.92,7,38.92,LONG,SPY
1,2020-01-31,321.73,2020-02-04,329.06,7.33,7.33,1,46.25,LONG,SPY
2,2020-01-31,321.73,2020-02-05,332.86,11.13,66.78,6,113.03,LONG,SPY
3,2020-02-21,333.48,2020-03-04,312.86,-20.62,-144.34,7,-31.31,LONG,SPY
4,2020-02-24,322.42,2020-03-04,312.86,-9.56,-19.12,2,-50.43,LONG,SPY
5,2020-02-24,322.42,2020-03-26,261.20,-61.22,-306.10,5,-356.53,LONG,SPY
6,2020-02-25,312.65,2020-03-26,261.20,-51.45,-205.80,4,-562.33,LONG,SPY
7,2020-02-25,312.65,2020-03-30,261.65,-51.00,-153.00,3,-715.33,LONG,SPY
8,2020-02-26,311.50,2020-03-30,261.65,-49.85,-348.95,7,-1064.28,LONG,SPY
9,2020-06-24,304.09,2020-07-02,312.23,8.14,56.98,7,-1007.30,LONG,SPY


In [9]:
# View merged trade log
strategies[symbols[1]].tlog

,entry_price,exit_price,pl_points,pl_cash,qty,cumul_total,direction,symbol,entry_date,exit_date
0,322.62,330.69,24.02,113.03,14,113.03,LONG,SPY,2020-01-27,2020-02-04
1,322.62,330.69,24.02,113.03,14,113.03,LONG,SPY,2020-01-31,2020-02-04
2,321.73,332.32,18.46,74.11,7,113.03,LONG,SPY,2020-01-31,2020-02-05
3,327.95,294.41,-91.40,-469.56,14,-356.53,LONG,SPY,2020-02-21,2020-03-04
4,327.95,294.41,-91.40,-469.56,14,-356.53,LONG,SPY,2020-02-24,2020-03-04
5,317.54,268.68,-173.23,-684.02,14,-715.33,LONG,SPY,2020-02-24,2020-03-26
6,317.54,268.68,-173.23,-684.02,14,-715.33,LONG,SPY,2020-02-25,2020-03-26
7,312.07,261.52,-152.30,-707.75,14,-1064.28,LONG,SPY,2020-02-25,2020-03-30
8,312.07,261.52,-152.30,-707.75,14,-1064.28,LONG,SPY,2020-02-26,2020-03-30
9,304.09,312.23,8.14,56.98,7,-1007.30,LONG,SPY,2020-06-24,2020-07-02


Summarize results - compare statistics between unmerged and merged view of trade log.

In [10]:
metrics = strategies[symbol].stats.index

df = pf.optimizer_summary(strategies, metrics)
df

,SPY,SPY_merged
start,2020-01-02,2020-01-02
end,2026-06-05,2026-06-05
beginning_balance,"$10,000.00","$10,000.00"
ending_balance,"$11,969.57","$11,969.57"
total_net_profit,"$1,969.57","$1,969.57"
gross_profit,"$5,006.76","$13,146.98"
gross_loss,"-$3,037.19","-$9,232.22"
profit_factor,1.65,1.42
return_on_initial_capital,19.70,19.70
annual_return_rate,2.84,2.84
